# Per-head Muon

Muon normally orthogonalizes a 2D parameter as one matrix. For an attention-style projection with weight shape `(num_heads * head_dim, in_features)`, that couples all output heads into the same Newton–Schulz iteration. This notebook implements two examples demonstrating how it can be implemented.

In [1]:
import torch
import torch.nn.functional as F

from emerging_optimizers.orthogonalized_optimizers.muon import Muon


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"device={device}, torch={torch.__version__}")

device=cuda, torch=2.8.0+cu129


## Implementation

### PerHeadMuonViaOrthogonalize

The `orthogonalize` of base class `OrthogonalizedOptimizer` is designed for such customization.

When `parameter.num_heads` is set, `orthogonalize()` temporarily reshapes only the momentum update to `(num_heads, head_dim, in_features)`, applies Muon's batched Newton–Schulz independently to each head, and reshapes the result back to 2D. When `num_heads is None`, it delegates to stock Muon.

In [2]:
class PerHeadMuonViaOrthogonalize(Muon):
    def orthogonalize(self, p, grad, **group_kwargs):
        num_heads = getattr(p, "num_heads", None)
        if num_heads is None:
            return super().orthogonalize(p, grad, **group_kwargs)
        if not isinstance(num_heads, int) or num_heads <= 0:
            raise ValueError(f"num_heads must be a positive integer, got {num_heads!r}")
        if grad.ndim != 2:
            raise ValueError(f"Expected a 2D parameter, got {grad.ndim}D")
        if grad.size(0) % num_heads != 0:
            raise ValueError(f"output dimension {grad.size(0)} must be divisible by num_heads={num_heads}")

        original_shape = grad.shape
        grad = grad.reshape(num_heads, grad.size(0) // num_heads, grad.size(1))
        return self.scaled_orthogonalize_fn(grad).reshape(original_shape)

### PerHeadMuonViaGrouping
This example shows a broad manipulation of parameter shapes and how they are preconditioned inside optimizer. Same technique can be used to customize other optimizers (`SOAP` for example) not just Muon.

A native 3D parameter is interpreted as `(num_heads, rows, columns)`. A flattened 2D parameter opts into per-head behavior by setting `parameter.num_heads`. `_group_heads` and `_ungroup_heads` provide the shape round trip around the inherited step.

`PerHeadMuonViaGrouping.orthogonalize` only removes the base class's 2D-only guard so that the parent's step can pass the grouped 3D momentum to Muon's existing batched Newton–Schulz implementation.

In [3]:
def _group_heads(x, num_heads):
    if num_heads is not None and (not isinstance(num_heads, int) or num_heads <= 0):
        raise ValueError(f"num_heads must be a positive integer, got {num_heads!r}")
    if x.ndim == 3:
        if num_heads is not None and x.size(0) != num_heads:
            raise ValueError(f"leading dimension {x.size(0)} must match num_heads={num_heads}")
        return x
    if x.ndim != 2:
        raise ValueError(f"PerHeadMuonViaGrouping expects a 2D or 3D parameter, got {x.ndim}D")
    if num_heads is None:
        return x
    if x.size(0) % num_heads != 0:
        raise ValueError(f"output dimension {x.size(0)} must be divisible by num_heads={num_heads}")
    return x.reshape(num_heads, x.size(0) // num_heads, x.size(1))


def _ungroup_heads(x, shape):
    return x.reshape(shape)


class PerHeadMuonViaGrouping(Muon):
    def orthogonalize(self, p, grad, **group_kwargs):
        if grad.ndim not in (2, 3):
            raise ValueError(f"PerHeadMuonViaGrouping expects a 2D or 3D parameter, got {grad.ndim}D")
        return self.scaled_orthogonalize_fn(grad)

    @torch.no_grad()
    def step(self, closure=None):
        if closure is not None:
            raise ValueError("closure is not supported")

        saved = []
        try:
            for group in self.param_groups:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    num_heads = getattr(p, "num_heads", None)
                    data, grad = p.data, p.grad
                    grouped_data = _group_heads(data, num_heads)
                    grouped_grad = _group_heads(grad, num_heads)
                    saved.append((p, data, grad))
                    p.data = grouped_data
                    p.grad = grouped_grad

            super().step()
        finally:
            for p, data, grad in saved:
                grouped_data = p.data
                p.data = data
                p.grad = grad
                if grouped_data.data_ptr() != data.data_ptr():
                    data.copy_(_ungroup_heads(grouped_data, data.shape))
        return None

## Check against independent Muon parameters

The next cell updates the same matrices in three representations:

1. one native 3D parameter optimized by `PerHeadMuonViaGrouping`;
2. one 2D parameter with a monkey-patched `num_heads` attribute optimized by `PerHeadMuonViaOrthogonalize`;
3. a list of independent 2D parameters optimized by stock `Muon`.

All three should produce the same per-head result. The `PerHeadMuonViaOrthogonalize` parameter, gradient, and momentum buffer remain 2D because grouping happens only inside `orthogonalize`.

In [4]:
torch.manual_seed(1)
num_heads, head_dim, in_features = 4, 6, 9
initial = torch.randn(num_heads, head_dim, in_features, device=device)

native_weight = torch.nn.Parameter(initial.clone())
orthogonalize_weight = torch.nn.Parameter(initial.flatten(0, 1).clone())
native_weight.num_heads = None
orthogonalize_weight.num_heads = num_heads
reference_weights = [torch.nn.Parameter(head.clone()) for head in initial]

assert native_weight.num_heads is None
assert orthogonalize_weight.num_heads is not None

settings = {
    "lr": 1e-2,
    "momentum": 0.9,
    "weight_decay": 0.01,
    "fp32_matmul_prec": "highest",
}
grouping_optimizer = PerHeadMuonViaGrouping([native_weight], **settings)
orthogonalize_optimizer = PerHeadMuonViaOrthogonalize([orthogonalize_weight], **settings)
reference_optimizer = Muon(reference_weights, **settings)

for _ in range(4):
    gradient = torch.randn_like(initial)
    native_weight.grad = gradient.clone()
    orthogonalize_weight.grad = gradient.flatten(0, 1).clone()
    for weight, head_gradient in zip(reference_weights, gradient):
        weight.grad = head_gradient.clone()

    grouping_optimizer.step()
    orthogonalize_optimizer.step()
    reference_optimizer.step()

reference = torch.stack([weight.detach() for weight in reference_weights])
torch.testing.assert_close(native_weight.detach(), reference, atol=1e-6, rtol=1e-6)
torch.testing.assert_close(
    orthogonalize_weight.detach().reshape_as(reference),
    reference,
    atol=1e-6,
    rtol=1e-6,
)
native_error = (native_weight.detach() - reference).abs().max().item()
orthogonalize_error = (orthogonalize_weight.detach().reshape_as(reference) - reference).abs().max().item()
print(f"maximum absolute difference, native 3D vs independent heads: {native_error:.3e}")
print(f"maximum absolute difference, orthogonalize-only vs independent heads: {orthogonalize_error:.3e}")
print(f"native parameter/state shapes: {native_weight.shape} / {grouping_optimizer.state[native_weight]['momentum_buffer'].shape}")
print(f"orthogonalize-only parameter/gradient shapes: {orthogonalize_weight.shape} / {orthogonalize_weight.grad.shape}")
print(f"orthogonalize-only momentum shape: {orthogonalize_optimizer.state[orthogonalize_weight]['momentum_buffer'].shape}")

maximum absolute difference, native 3D vs independent heads: 0.000e+00
maximum absolute difference, orthogonalize-only vs independent heads: 0.000e+00
native parameter/state shapes: torch.Size([4, 6, 9]) / torch.Size([4, 6, 9])
orthogonalize-only parameter/gradient shapes: torch.Size([24, 9]) / torch.Size([24, 9])
orthogonalize-only momentum shape: torch.Size([24, 9])
